In [4]:
!pip install konlpy JPype1

   ---------------------------------------- 0.0/19.4 MB ? eta -:--:--
   -- ------------------------------------- 1.3/19.4 MB 10.3 MB/s eta 0:00:02
   ------- -------------------------------- 3.7/19.4 MB 11.4 MB/s eta 0:00:02
   ------------ --------------------------- 6.3/19.4 MB 11.6 MB/s eta 0:00:02
   ----------------- ---------------------- 8.7/19.4 MB 11.6 MB/s eta 0:00:01
   ---------------------- ----------------- 11.0/19.4 MB 11.7 MB/s eta 0:00:01
   --------------------------- ------------ 13.4/19.4 MB 11.7 MB/s eta 0:00:01
   -------------------------------- ------- 15.7/19.4 MB 11.7 MB/s eta 0:00:01
   ------------------------------------- -- 18.4/19.4 MB 11.8 MB/s eta 0:00:01
   ---------------------------------------  19.4/19.4 MB 11.8 MB/s eta 0:00:01
   ---------------------------------------- 19.4/19.4 MB 10.3 MB/s  0:00:01

   ---------------------------------------- 0/2 [JPype1]
   ---------------------------------------- 0/2 [JPype1]
   -------------------- --------

In [ ]:
# !pip install konlpy JPype1 <-- 터미널/셀에서 한 번만 실행
from flask import Flask, request, jsonify
import joblib
from konlpy.tag import Okt
import os

app = Flask(__name__)

# Java 경로 설정
os.environ['JAVA_HOME'] = r'C:\Program Files\Java\jdk-21'

# 모델 로드 (서버 켤 때 한 번만)
# 주피터에서 저장한 파일 이름과 동일해야함.
try:
    model = joblib.load('analyzer_meokbap.pkl')
    vectorizer = joblib.load('tfidf.pkl')
    okt = Okt()
    print('모델 로드 완료!')
except:
    print('모델 로드 실패...')

# 분석 함수
def predict_smart(text):
    # 형태소 분석 및 전처리
    clean_text = ' '.join(okt.morphs(okt.normalize(str(text)), stem=True))

    # 강력 교정 키워드
    correction_keywords = ['우울', '울적', '스트레스', '홧김', '꿀꿀', '슬프다', '힘들다', '답답']

    if any(word in text for word in correction_keywords) or any(word in clean_text for word in correction_keywords):
        return 1, 100.0

    # 벡터화 및 예측
    vec = vectorizer.transform([clean_text])
    pred = model.predict(vec)[0]
    prob = model.predict_proba(vec)[0]
    return int(pred), float(max(prob) * 100)

@app.route('/predict', methods=['POST'])
def predict():
    data = request.json
    user_text = data.get('text', '')

    # 함수 실행
    label, confidence = predict_smart(user_text)

    return jsonify({
        'label': label,
        'confidence': f"{confidence:.2f}%",
        'result': '충동/보상 지출' if label == 1 else '일반 지출'
    })

if __name__ == "__main__":
    # Java 서버(8087)와 충돌 일어나지 않게 5000번 포트 유지
    app.run(port=5000)

모델 로드 완료!
 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
